# Webpage Content Pre-Processing

## Overview
Due to the characterstics of web archive data, the pre-processing of webpage content is an important step in reducing noise and duplicate content. Particularly when that data is used downstream in features such as semantic search. The quality of the results from semantic search are highly dependent on the quality of the data and how effectively it has been 'cleaned'.

### Data Cleaning Process
1. Load the raw HTML content extracted from the web archive payloads (removing the HTML boilerplate and focused on main content part).
2. Embedding the content and conduct similarity comparison for addressing semantic differences between snapshots.
3. Filter the new content (new_or_changed) webpages for building the vector database.


> ***Note***
>
> *This notebook builds on the same Python packages used in the previous notebooks.*

## Environment Setup

### Installing Required Python Packages

In [ ]:
# Colab bootstrap: install the packages required for preprocessing web archive HTML contents in this notebook. You can skip this cell if you are running the notebook in your local environment and have already installed the required packages.
%pip install --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ wa-nlnz-toolkit
%pip install -q tqdm fastparquet pyarrow # additional dependencies for this notebook

In [ ]:
import os
import wa_nlnz_toolkit as want
import pandas as pd  # order matters, don't change the import order
import pyarrow as pa
from glob import glob
from tqdm import tqdm

# Configure environment based on execution context
try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Set output directory based on execution environment
res_folder = "/content" if IN_COLAB else "./"

## 1. Preprocessing with boilerplate removal

In [ ]:
# Define S3 bucket and folder containing the archive data
bucket_name = "ndha-public-data-ap-southeast-2"
folder_prefix = "IIPC2026WAC/sample-data/covid19.govt.nz/"

# List all files in the S3 bucket folder
all_files = want.list_s3_files(bucket_name, folder_prefix)

# Filter for CDX index files
cdx_files = [f for f in all_files if f.endswith(".cdx")]

# Display the total number of crawls in the dataset
print(f"In total, there are {len(cdx_files)} crawls in the sample dataset.")


# Helper function to locate a specific WARC file in the S3 bucket
def find_warc_file_path(warc_file):
    """Find the full S3 path for a given WARC filename.

    Args:
        warc_file (str): The WARC filename to search for

    Returns:
        str: Full S3 path if found, None otherwise
    """
    for s3_file in all_files:
        if warc_file in s3_file:
            warc_file_path = f"s3://{bucket_name}/" + s3_file
            return warc_file_path
    return None

In [ ]:
# Process each CDX file
def extract_and_save_content(cdx_file, content_dir, boilerplate_removal_method):
    # Load CDX file data into a DataFrame
    df_cdx = want.load_cdx_file_from_s3(bucket_name, cdx_file)

    # Extract date from filename
    dt = cdx_file.split("/")[-2].split("_")[0]

    # Initialize lists to store content and URLs
    contents = []
    urls = []

    # Process each entry in the CDX file
    for idx in tqdm(range(len(df_cdx)), desc=f"Extracting contents for {dt}"):
        # Get WARC file and offset information
        warc_file = df_cdx.iloc[idx]["g"]
        offset = int(df_cdx.iloc[idx]["V"])

        # Extract HTML payload and text content
        html_payload = want.extract_payload(find_warc_file_path(warc_file), offset)

        # Default boilerplate removal and text extraction approach
        # Try other approaches and compare results if needed
        html_content = boilerplate_removal_method(html_payload)

        # Construct access URL for this capture
        url = "https://ndhadeliver.natlib.govt.nz/webarchive/{}/{}".format(
            df_cdx.iloc[idx]["b"], df_cdx.iloc[idx]["a"]
        )

        # Log URLs with no content extracted
        if (html_content == []) or (html_content == ""):
            print(f"No content extracted from: {url}")
        else:
            contents.append(html_content)
            urls.append(url)

    # Joint the content list into a single string for each entry (if it's a list of paragraphs)
    contents = [
        "\n\n".join(content) if isinstance(content, list) else content
        for content in contents
    ]

    # Store the data into a pandas DataFrame
    # NOTE: In practice, the Apache Parquet will be used for better performance and storage efficiency
    df_content = pd.DataFrame({"url": urls, "content": contents})

    # Ensure all content is string-type to avoid Arrow extension errors
    df_content["url"] = df_content["url"].astype(str)
    df_content["content"] = df_content["content"].astype(str)

    # Save the DataFrame to a Parquet file (optional, for intermediate storage)
    df_content.to_parquet(
        os.path.join(content_dir, f"covid19_content_{dt}.parquet"),
        index=False,
        engine="pyarrow",  # Explicitly set the engine
    )

### Default extraction

Extract text content from an HTML payload by parsing its main content elements.
The default extraction function extracts text from ``<p>`` and ``<li>`` elements within each ``<section>`` and returns the collected text as a list of strings. When multiple sections contain text, a section separator string is inserted between them.

In [ ]:
# Create directories for storing extracted content
content_dir = os.path.join(res_folder, "sample_data/covid19_corpus/raw_default")

# Ensure directories exist
os.makedirs(content_dir, exist_ok=True)

extract_and_save_content(
    cdx_files[0],
    content_dir,
    boilerplate_removal_method=want.extract_content_html,
)

### markitdown

https://github.com/microsoft/markitdown

Use markitdown to extract the main content from the HTML payload. Markitdown is a tool that converts HTML documents into Markdown format, while also providing options to extract specific content based on CSS selectors or other criteria. 

In [ ]:
!pip install -q markitdown

In [ ]:
import io
from markitdown import MarkItDown

md = MarkItDown()


def extract_content_with_markitdown(html_payload):
    # Check if it's a string, if so, convert to bytes
    if isinstance(html_payload, str):
        html_payload = html_payload.encode("utf-8")

    stream = io.BytesIO(html_payload)
    result = md.convert_stream(stream, extension=".html")

    # Access the converted text
    return result.text_content

In [ ]:
# Create directories for storing extracted content
content_dir = os.path.join(res_folder, "sample_data/covid19_corpus/raw_markitdown")

# Ensure directories exist
os.makedirs(content_dir, exist_ok=True)

extract_and_save_content(
    cdx_files[0],
    content_dir,
    boilerplate_removal_method=extract_content_with_markitdown,
)

### Trafilatura

https://github.com/adbar/trafilatura

Trafilatura is a Python library for web scraping and content extraction. It provides functions to extract the main content from HTML documents, while also removing boilerplate elements such as navigation menus, advertisements, and other non-essential content.

In [ ]:
# BUGFIX default trafilatura installation by installing lxml_html_clean explicitly
!pip -q install trafilatura lxml_html_clean

In [ ]:
from trafilatura import extract


def extract_content_trafilatura(html_payload):
    # Extract content using trafilatura
    html_content = extract(html_payload)

    return html_content

In [ ]:
# Create directories for storing extracted content
content_dir = os.path.join(res_folder, "sample_data/covid19_corpus/raw_trafilatura")

# Ensure directories exist
os.makedirs(content_dir, exist_ok=True)

extract_and_save_content(
    cdx_files[0],
    content_dir,
    boilerplate_removal_method=extract_content_trafilatura,
)

### jusText

https://github.com/miso-belica/justext

jusText is a Python library for boilerplate removal and content extraction from HTML documents. It uses a combination of heuristics and machine learning techniques to identify and extract the main content from web pages, while filtering out boilerplate elements such as headers, footers, navigation menus, and advertisements.

In [ ]:
!pip install -q justext

In [ ]:
import justext


def extract_content_justext(html_payload):
    # Extract content using jusText
    paragraphs = justext.justext(html_payload, justext.get_stoplist("English"))
    html_content = "\n\n".join(
        [para.text for para in paragraphs if not para.is_boilerplate]
    )

    return html_content

In [ ]:
# Create directories for storing extracted content
content_dir = os.path.join(res_folder, "sample_data/covid19_corpus/raw_justext")

# Ensure directories exist
os.makedirs(content_dir, exist_ok=True)

extract_and_save_content(
    cdx_files[0],
    content_dir,
    boilerplate_removal_method=extract_content_justext,
)

### Compare the results of different boilerplate removal methods

We can compare the results of different boilerplate removal methods by applying them to the same HTML content and evaluating the extracted text. We used the structured results from the markitdown extraction as the reference, and compared the results from the default extraction, Trafilatura, and jusText methods against it. A python function in the wa-nlnz-toolkit is implemented to conduct the comparison, and the results are shown in the HTML table.

In [ ]:
# Get all URLs from the 2020-03-19 crawl using the default extraction method
df_default = pd.read_parquet(
    "./sample_data/covid19_corpus/raw_default/covid19_content_2020-03-19.parquet"
)
print(df_default["url"])

In [ ]:
# Select one URL for comparison
url = "https://ndhadeliver.natlib.govt.nz/webarchive/20200318051641/https://covid19.govt.nz/"
# url = "https://ndhadeliver.natlib.govt.nz/webarchive/20200318052121/https://covid19.govt.nz/help-and-advice/for-everyone/vulnerable-people/"

In [ ]:
# Load extracted content tables
parquet_paths = {
    "default": "./sample_data/covid19_corpus/raw_default/covid19_content_2020-03-19.parquet",
    "trafilatura": "./sample_data/covid19_corpus/raw_trafilatura/covid19_content_2020-03-19.parquet",
    "justext": "./sample_data/covid19_corpus/raw_justext/covid19_content_2020-03-19.parquet",
    "markitdown": "./sample_data/covid19_corpus/raw_markitdown/covid19_content_2020-03-19.parquet",
}

dfs = {name: pd.read_parquet(path) for name, path in parquet_paths.items()}

# Keep explicit dataframe names for readability / downstream cells
df_default = dfs["default"]
df_trafilatura = dfs["trafilatura"]
df_justext = dfs["justext"]
df_markitdown = dfs["markitdown"]

# Compare content extraction results for the selected URL across different methods
html_output, output_path = want.build_content_comparison_html(url, dfs)

## 2. Preprocessing all snapshots

Now we'll extend the text extraction process to all crawls in the dataset.

By default we will use the default extraction method provided in the wa-nlnz-toolkit. However, we can easily switch to other methods by changing the `boilerplate_removal_method` parameter in the `extract_and_save_content` function.

In [ ]:
# Create directories for storing extracted content
content_dir = os.path.join(res_folder, "sample_data/covid19_corpus/raw_default")

# Ensure directories exist
os.makedirs(content_dir, exist_ok=True)

for cdx_file in cdx_files:
    extract_and_save_content(
        cdx_file,
        content_dir,
        boilerplate_removal_method=want.extract_content_html,
    )

Entire processing will take around 1 hour to finish. We have also prepared all processed data files in AWS S3 for this workshop!

Run the following cell to download the extracted corpus dataset to local.

Replace "default" with either "justext", "trafilatura", or "markitdown".

In [ ]:
corpus_dataset_name = "raw_default"

uri_prefix = f"IIPC2026WAC/covid19_corpus/{corpus_dataset_name}/"
uri_data_all = want.list_s3_files(bucket_name, uri_prefix)
for uri_data in uri_data_all:
    want.download_s3_file(
        bucket_name,
        uri_data,
        os.path.join(res_folder, uri_data.replace("IIPC2026WAC", "sample_data")),
    )

## 3. Deduplication using embedding similarity

After removing the boilerplate in the raw HTML payload, it is common that very similar contents were captured in the crawl due to URL aliases, repeated pagination, or frequent re‑crawls of slowly changing pages. To address this, we apply deduplication based on embedding similarity rather than relying solely on exact or near‑exact text matching.

Embedding‑based deduplication treats each document (or document chunk) as a semantic vector and removes duplicates by comparing their positions in embedding space. This approach is robust to superficial variations such as reordered sentences, minor wording changes, or injected timestamps that would defeat hash‑based or n‑gram‑based methods.

In [ ]:
!pip install sentence-transformers scikit-learn

In [ ]:
import os
import re


# Reuse the extract_date function
def extract_date(fname):
    """Extract date in YYYY-MM-DD format from a filename.

    Args:
        fname: Filename string containing a date

    Returns:
        Date string in YYYY-MM-DD format or None if not found
    """
    match = re.search(r"(\d{4}-\d{2}-\d{2})", fname)
    return match.group(1) if match else None


# Configuration for vector database
embedding_model = "all-MiniLM-L6-v2"  # Pre-trained embedding model
input_content_dir = os.path.join(
    res_folder, "sample_data/covid19_corpus/raw_default"
)  # Directory with raw text files

### Similarity comparison and content filtering

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd


# -------------------------
# Load snapshot
# -------------------------
files_content = sorted(
    [f for f in os.listdir(input_content_dir) if f.endswith(".parquet")]
)

snapshot_pages = []
snapshot_urls = []
for file_content in files_content:
    # Load content file
    content_file_path = os.path.join(input_content_dir, file_content)

    lines = pd.read_parquet(content_file_path)["content"].tolist()
    urls = pd.read_parquet(content_file_path)["url"].tolist()

    snapshot_pages.append(lines)
    snapshot_urls.append(urls)

# -------------------------
# Model
# -------------------------
model = SentenceTransformer(embedding_model)


def embed_snapshots(snapshot_pages):
    """
    Embedding function for snapshots.
    """
    return model.encode(
        snapshot_pages, convert_to_numpy=True, normalize_embeddings=True
    )


# -------------------------
# Snapshot embeddings
# -------------------------
snapshot_embs = []
for snapshot_page in snapshot_pages:
    snapshot_embs.append(embed_snapshots(snapshot_page))


# -------------------------
# Semantic diff between snapshots
# -------------------------
results = []

for i in range(len(snapshot_embs)):
    prev = snapshot_embs[i - 1] if i > 0 else []
    curr = snapshot_embs[i]

    status_list = []

    for idx, emb in enumerate(curr):
        if len(prev) == 0:
            status_list.append(("new_or_changed", idx))
            continue

        sims = cosine_similarity([emb], prev)[0]
        best_sim = sims.max()

        if best_sim > 0.95:
            status = "unchanged"
        elif best_sim > 0.85:
            status = "slightly_updated"
        else:
            status = "new_or_changed"

        status_list.append((status, idx))

    results.append(status_list)

### Display the results and analyze the effectiveness of the deduplication process

Run the following cell to plot the status of pages captured from the covid19.govt.nz web archive over time. 

In [ ]:
from collections import Counter
import pandas as pd


def count_statuses(data):
    """
    Counts the number of each status in a list of tuples,
    ignoring the second column.
    """
    # Generator expression extracts the first element (status) from each tuple
    return Counter(row[0] for row in data)


# Show variations of page status among snapshots
snapshot_dt = [extract_date(filename) for filename in files_content]

df_status_counts = pd.DataFrame(
    [count_statuses(result) for result in results], index=snapshot_dt
)
# df_status_counts = df_status_counts[["new_or_changed", "slightly_updated", "unchanged"]]

df_status_counts.plot.bar(
    stacked=True,
    title="Web archive (covid19.govt.nz) page status",
    figsize=(10, 6),
    ylabel="Number of pages",
)

As seen from the figure above, most pages remain unchanged across crawl snapshots, with smaller portions reflecting new or slightly updated content during periods of active revision. This pattern highlights the prevalence of semantically redundant pages in web crawls and motivates the use of embedding‑based deduplication to remove near‑duplicate content.

To achieve better semantic search results, we will filter new_or_changed webpages for building the downstream vector database (for semantic search).

### Filter new or changed webpages for building the vector database

In [ ]:
def extract_changed_pages(
    snapshot_pages_all, snapshot_status_all, snapshot_urls_all, filter
):
    changed_pages = []

    # results[0] -> snapshots[1]
    for i, snapshot_status in enumerate(snapshot_status_all):
        snapshot_id = i
        snapshot_pages = snapshot_pages_all[snapshot_id]
        snapshot_urls = snapshot_urls_all[snapshot_id]

        for status, page_idx in snapshot_status:
            if status == filter:
                # ignore pages with little information (character length < 100)
                if len(snapshot_pages[page_idx]) < 100:
                    continue

                changed_pages.append(
                    {
                        "snapshot_id": snapshot_id,
                        "page_idx": page_idx,
                        "url": snapshot_urls[page_idx],
                        "text": snapshot_pages[page_idx],
                    }
                )
    return changed_pages


changed_pages = extract_changed_pages(
    snapshot_pages, results, snapshot_urls, "new_or_changed"
)

In [ ]:
# save to local as parquet for better readability and downstream processing
df_changed_pages = pd.DataFrame(changed_pages)
df_changed_pages.to_parquet(
    os.path.join(res_folder, "sample_data", "preprocessed_corpus.parquet"),
    index=False,
    engine="pyarrow",
)

### Citation

**Title:** Webpage content preprocessing (Jupyter notebook)  
**Author:** Yizhe Zhan, Ben O'Brien  
**Affiliation:** National Library of New Zealand, Wellington  
**Last updated:** 2026‑04‑14  

**Contact:** yizhe.git@gmail.com  
**Repository:** https://github.com/NLNZDigitalPreservation/wa-nlnz-toolkit